In [11]:
#!/usr/bin/env python3
import os
import pandas as pd

def reverse_readline(fh, buf_size=8192):
    """
    A generator that returns the lines of a file in reverse order.
    Reads file by blocks from the end. (Works for text files opened in binary mode.)
    """
    segment = None
    offset = 0
    fh.seek(0, os.SEEK_END)
    position = fh.tell() # position where the file was read
    file_size = fh.tell()
    while offset < file_size:
        # Incrementally decrease the position of read
        offset = min(file_size, offset + buf_size)
        position = file_size - offset
        fh.seek(position)

        # Read a buffer size of data
        buffer = fh.read(min(buf_size, offset))

        # split the buffer by newline
        lines = buffer.split(b'\n')

        # The first segment of the current buffer is likely a partial line, so add it to the previous segment.
        if segment is not None:
            if buffer[-1] != ord(b'\n'):
                # If the last char isn't a newline, then the last element is partial.
                lines[-1] += segment
            else:
                lines.append(segment)
        segment = lines[0]

        for line in reversed(lines[1:]):
            line = line.decode('utf-8', errors='replace')
            yield line.strip(), position
    # yield the last remaining segment
    if segment is not None:
        segment = segment.decode('utf-8', errors='replace')
        yield segment.strip(), position

def forward_read_lines(header, date_time, log_file_path, start_offset, num_lines=50):
    """
    Open the file in forward (normal) mode, seek to start_offset, and read num_lines.
    Returns a list of strings (lines).
    """
    lines = []
    with open(log_file_path, "r", encoding="utf-8", errors="replace") as f:
        f.seek(start_offset - 81920) # make sure enough lines are read

        marker_found = False
        count = 0
        while (count < num_lines):
            line = f.readline()

            if header in line and date_time in line:
                marker_found = True
                continue

            if marker_found:
                lines.append(line.rstrip("\n"))
                count = count + 1
    return lines

def parse_diagnostics_line(line):
    """
    Given a diagnostics line of the format:
    Negative cation_vr/cec_cation_vr diagnostics: <latitude> <longitude> <i> <j> <k> <value> <date> <time>
    Parse and return latitude, longitude, and datetime as strings.
    """
    parts = line.split()
    # Assuming the line splits as:
    # 0: Negative
    # 1: cation_vr/cec_cation_vr
    # 2: diagnostics:
    # 3: <latitude>
    # 4: <longitude>
    # 5: <i>
    # 6: <j>
    # 7: <k>
    # 8: <value>
    # 9: <date_time>
    try:
        latitude = float(parts[3])
        longitude = float(parts[4])
        date_time = parts[9]
    except IndexError:
        raise ValueError("Diagnostics line does not have the expected format")
    return latitude, longitude, date_time

def parse_table_line(line, header):
    """
    Given a table line starting with <i> <j> <k> ..., parse it into tokens.
    """
    tokens = line.split()

    if header == "Post-reaction cec cation":
        index = ['c','j','icat','cec_cation_vr', 'meq cec_cation_vr', 
                 '-cec_cation_flux_vr * dt', '-cec_cation_flux2_vr * dt', 
                 'background_cec_vr * dt']
    elif header == "Post-reaction cation":
        index = ['c','j','icat','cation_vr','mol cation_vr',
                 'background_flux_vr * dt', 'primary_cation_flux_vr * dt', 
                 'cec_cation_flux_vr * dt', 'cec_cation_flux2_vr * dt', 
                 '-secondary_cation_flux_vr * dt', 
                 '-cation_uptake_vr * dt', 'cation_infl_vr * dt', 
                 '-cation_leached_vr * dt', 'cation_runoff_vr * dt']

    tokens = pd.Series(tokens, index = index).astype(float)

    # You may want to convert tokens to appropriate types, e.g. int or float.
    # For now, we simply return the tokens.
    return tokens

def main():
    # Change this to your error log file path
    log_file_path = "/gpfs/wolf2/cades/cli185/proj-shared/ywo/E3SM/output/20250328_UIEF_ICB20TRCNPRDCTCBC_3year_rmethod1_testSSAerw/run/fort.100"

    diagnostics_line = None

    # Open the file in binary mode for reverse reading.
    with open(log_file_path, "rb") as fh:
        # Read lines in reverse
        for line, _ in reverse_readline(fh):
            # Look for diagnostics line
            if diagnostics_line is None and "Negative cation_vr/cec_cation_vr diagnostics:" in line:
                diagnostics_line = line
                # We found the diagnostics line
                break

    if diagnostics_line is None:
        print("Diagnostics line not found in file.")
        return

    # Parse the diagnostics line to extract latitude, longitude, and datetime
    try:
        latitude, longitude, date_time = parse_diagnostics_line(diagnostics_line)
    except ValueError as ve:
        print("Error parsing diagnostics line:", ve)
        return

    print("Diagnostics data:")
    print("Latitude:", latitude)
    print("Longitude:", longitude)
    print("Datetime:", date_time)

    # Diagnostic line is found. Open the file in binary mode for reverse reading for actual info. 
    collected_lines = {"Post-reaction cec cation": [],
                       "Post-reaction cation": []}  # Will collect lines from diagnostics upward to the key marker
    with open(log_file_path, "rb") as fh:
        # Read lines in reverse
        for line, position in reverse_readline(fh):
            if "Post-reaction cec cation" in line:
                collected_lines["Post-reaction cec cation"] = forward_read_lines("Post-reaction cec cation", date_time, log_file_path, position)
                continue
            if "Post-reaction cation" in line:
                collected_lines["Post-reaction cation"] = forward_read_lines("Post-reaction cation", date_time, log_file_path, position)
                break

    # Reverse collected_lines so that they are in original order (from "Post-reaction cec cation" down to diagnostics)
    for key in collected_lines.keys():
        table_lines = collected_lines[key]

        table_lines.reverse()

        for i, line in enumerate(table_lines):
            stripped = line.lstrip()
            if stripped and stripped[0].isdigit():
                table_lines[i] = parse_table_line(line, key)
            else:
                print("Table line not found after 'Post-reaction cec cation' marker.")
                return

        table_lines = pd.DataFrame(table_lines)
        table_lines['c'] = table_lines['c'].astype(int)
        table_lines['j'] = table_lines['j'].astype(int)
        table_lines['icat'] = table_lines['icat'].astype(int)
        table_lines = table_lines.set_index(['c','j','icat']).sort_index()

        collected_lines[key] = table_lines
    
    return collected_lines


if __name__ == '__main__':
    collected_lines = main()


Diagnostics data:
Latitude: 40.06278
Longitude: -88.19610999999998
Datetime: 1939-04-29_20:00:00


In [13]:
collected_lines['Post-reaction cation']

cation_vr  mol cation_vr  background_flux_vr * dt  \
c j  icat                                                         
1 1  1     1.734442e+01   1.126451e-03             1.072581e-03   
     2     1.422263e-01   1.523150e-05             8.255962e-04   
     3     7.657854e-03   8.670149e-07             5.337420e-05   
     4     5.158862e-02   3.434428e-06             0.000000e+00   
     5     2.650561e-02   2.557139e-06             7.565299e-05   
  2  1     2.381791e+00   1.565074e-04             2.321164e-04   
     2     2.460604e-01   2.666143e-05             1.668984e-04   
     3     1.560439e-02   1.787496e-06             3.746873e-05   
     4     4.613873e-02   3.107742e-06             0.000000e+00   
     5     3.169635e-03   3.093888e-07             6.492859e-05   
  3  1     7.460478e-01   5.028235e-05             0.000000e+00   
     2     3.843605e-01   4.271674e-05             3.136420e-05   
     3     2.946841e-02   3.462364e-06             1.806788e-05   
     4     6.442431e-02   4.450890e-06             0.000000e+00   
     5     2.609648e-04   2.612732e-08             6.011704e-05   
  4  1     1.251632e+00   8.686296e-05             4.040310e-06   
     2     8.271463e-01   9.465654e-05             0.000000e+00   
     3     6.281994e-02   7.600154e-06             0.000000e+00   
     4     1.053725e-01   7.496063e-06             0.000000e+00   
     5     6.172759e-05   6.363576e-09             5.553847e-05   
  5  1     1.980454e+00   1.379413e-04             0.000000e+00   
     2     1.310272e+00   1.504878e-04             0.000000e+00   
     3     1.321221e-01   1.604250e-05             0.000000e+00   
     4     1.412938e-01   1.008790e-05             0.000000e+00   
     5     8.206870e-05   8.491238e-09             5.275538e-05   
  6  1     1.483163e+00   1.020412e-04             6.945298e-05   
     2     1.073049e+00   1.217352e-04             3.302980e-05   
     3     2.280128e-01   2.734718e-05             6.932600e-06   
     4     1.585667e-01   1.118269e-05             0.000000e+00   
     5     3.026914e-03   3.093503e-07             2.489725e-06   
  7  1     5.365759e-01   3.732937e-05             0.000000e+00   
     2     3.756403e-01   4.309254e-05             0.000000e+00   
     3     1.770823e-01   2.147641e-05             0.000000e+00   
     4     6.392945e-02   4.558990e-06             0.000000e+00   
     5     6.223262e-05   6.431344e-09             0.000000e+00   
  8  1     5.770971e-02   4.083528e-06             0.000000e+00   
     2     3.517965e-02   4.104768e-06             0.000000e+00   
     3     3.764355e-02   4.643487e-06             0.000000e+00   
     4     1.777263e-02   1.289100e-06             0.000000e+00   
     5     1.067418e-06   1.121981e-10             0.000000e+00   
  9  1     5.547097e-03   4.203460e-07             0.000000e+00   
     2     2.890493e-03   3.611799e-07             0.000000e+00   
     3     7.430579e-03   9.815918e-07             0.000000e+00   
     4     4.146952e-03   3.221205e-07             0.000000e+00   
     5     9.018034e-09   1.015119e-12             0.000000e+00   
  10 1     4.727945e-03   2.900227e-07             9.372389e-09   
     2     2.342991e-03   2.369958e-07             4.755254e-09   
     3     6.808260e-03   7.280525e-07             5.405206e-09   
     4     4.971774e-03   3.126217e-07             0.000000e+00   
     5     7.396943e-09   6.740247e-13             3.032977e-14   

           primary_cation_flux_vr * dt  cec_cation_flux_vr * dt  \
c j  icat                                                         
1 1  1                      -98.598076             6.680964e+01   
     2                        0.001690            -5.039883e-02   
     3                        0.000000            -1.226691e-02   
     4                        0.000000            -1.053944e-02   
     5                        0.000750            -2.321063e-02   
  2  1                        0.001221            -1.505826e

In [12]:
collected_lines['Post-reaction cec cation']

cec_cation_vr  meq cec_cation_vr  -cec_cation_flux_vr * dt  \
c j  icat                                                               
1 1  1         -1.280023          -0.004301             -6.680964e+01   
     2          3.844515           0.021301              5.039883e-02   
     3          0.035136           0.000103              1.226691e-02   
     4         23.349372           0.040211              1.053944e-02   
     5         88.071184           0.659385              2.321063e-02   
  2  1       5247.250294          17.631201              1.505826e+00   
     2        465.247737           2.577774             -1.335871e-02   
     3          3.758966           0.011009             -7.619864e-04   
     4        166.306230           0.286403             -7.006149e-04   
     5        136.021349           1.018386              2.340200e-03   
  3  1       5610.507813          18.895059              1.637937e-02   
     2        647.917343           3.598124             -1.540734e-02   
     3          6.696137           0.019657             -1.352326e-03   
     4        237.662069           0.410227             -2.033206e-03   
     5         59.490324           0.446424              2.141678e-04   
  4  1       5210.124415          17.587027             -2.577340e-02   
     2        933.238442           5.194543             -1.791763e-02   
     3         16.322677           0.048026             -1.412834e-03   
     4        357.827688           0.619065             -2.305107e-03   
     5         63.422164           0.477025              2.707629e-05   
  5  1       5230.706595          17.697228             -9.158828e-03   
     2        932.666334           5.203333             -6.423366e-03   
     3         26.674754           0.078665             -7.894121e-04   
     4        392.271903           0.680221             -8.076612e-04   
     5         60.203245           0.453858              1.256382e-05   
  6  1       4930.348707          16.681017             -1.762975e-03   
     2       1042.311799           5.815043             -1.311521e-03   
     3         27.055298           0.079787             -2.829746e-04   
     4        360.144695           0.624511             -2.126998e-04   
     5         42.542527           0.320718             -1.919210e-06   
  7  1       4604.007415          15.505369              1.385720e-04   
     2        782.931553           4.347908              9.660338e-05   
     3         96.786912           0.284119              4.426102e-05   
     4        366.794861           0.633122              1.842788e-05   
     5         41.788032           0.313584              1.751536e-08   
  8  1       3615.338220          12.064930              5.307377e-05   
     2        570.162122           3.137507              3.381664e-05   
     3         56.848053           0.165359              2.567526e-05   
     4        283.039930           0.484107              1.172522e-05   
     5         33.208898           0.246937              2.838733e-09   
  9  1       2323.173597           7.614201              9.730579e-06   
     2        348.412162           1.882983              5.971377e-06   
     3         29.734790           0.084946              6.032087e-06   
     4        181.434694           0.304776              2.798320e-06   
     5         22.100453           0.161398              1.876451e-10   
  10 1       2282.859088           7.635617             -5.691237e-07   
     2        337.671256           1.862385             -2.647511e-07   
     3         27.677245           0.080691             -8.947560e-07   
     4        171.684878           0.294317             -8.128429e-07   
     5         20.616201           0.153649             -8.330147e-13   

           -cec_cation_flux2_vr * dt  background_cec_vr * dt  
c j  icat                                                     
1 1  1                           0.0            0.000000e+00  
     2                   